Using the DATA_GOV_API_KEY, retrieve data from `https://data.sfgov.org/resource/wg3w-h783`, SF Police Department Report Data.

complete store_to_gcs() to store retrieved data to GCS, and retrieve_data_from_gcs() to retrieve data from GCS.

- store_to_gcs(service_account_key, project_id, bucket_name, file_name, data) : upload data as file_name on the given gcs.
- retrieve_data_from_gcs(service_account_key, project_id, bucket_name, file_name, key_list) : return a list of list which includes values corresponding to keys in key_list.
  If

```
key_list = ['incident_datetime', 'report_datetime', 'incident_code',
            'incident_category', 'incident_description', 'latitude',
            'longitude', 'police_district']
```

This should return

```
[['2025-08-26T23:17:00.000',
  '2025-08-26T23:17:00.000',
  '07041',
  'Recovered Vehicle',
  'Vehicle, Recovered, Auto',
  None,
  None,
  'Out of SF'],
 ['2025-08-27T00:37:00.000',
  '2025-08-27T00:37:00.000',
  '04134',
  'Assault',
  'Battery',
  '37.78041458129883',
  '-122.44901275634766',
  'Park'],....]
```

If a key doesn't exist, its value should be `None`


In [16]:
import datetime
import json
import os

from dotenv import load_dotenv, find_dotenv
from google.oauth2 import service_account
from google.cloud import storage
import requests

In [17]:
env_path = find_dotenv()
print("Loaded .env path:", env_path)

load_dotenv(env_path)  # makes sure you're using this one

Loaded .env path: /Users/tomtranjr/Library/CloudStorage/GoogleDrive-ttran48@dons.usfca.edu/My Drive/fm1/msds692_data_acquisition_02/msds692_data_acquisition_2025/.env


True

In [13]:
load_dotenv()

True

In [18]:
api_key = os.getenv("DATA_GOV_API_KEY")

- Typically authorization in a header uses the format of..

```
headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
```

However, based on the [documentaiton](https://api.data.gov/docs/developer-manual/), I followed the following format.

Also, the data specific meta data and details can be searchable. (Ex.[link](https://data.sfgov.org/Public-Safety/Police-Department-Incident-Reports-2018-to-Present/wg3w-h783/about_data))


In [19]:
url = "https://data.sfgov.org/resource/wg3w-h783"
header = {"X-Api-Key": api_key}

response = requests.get(url, headers=header, timeout=10)

In [5]:
data = response.json()

Create store_to_gcs(service_account_key, project_id, bucket_name, file_name, data) which upload data as file_name on the given gcs.


In [ ]:
def store_to_gcs(
    service_account_key: str,
    project_id: str,
    bucket_name: str,
    file_name: str,
    data: str,
) -> None:
    try:
        credentials = service_account.Credentials.from_service_account_file(
            service_account_key,
        )
        client = storage.Client(project=project_id, credentials=credentials)
        bucket = client.bucket(bucket_name)
        blob = bucket.blob(file_name)

        blob.upload_from_string(data)
        print(f"✅ Successfully uploaded '{file_name}' to bucket '{bucket_name}'.")
    except BaseException as e:
        print(f"❌ Failed to upload '{file_name}' to bucket '{bucket_name}': {e}")

In [12]:
service_account_key = os.getenv("GCP_SERVICE_ACCOUNT_KEY")
project_id = os.getenv("GCP_PROJECT_ID")
bucket_name = os.getenv("GCP_BUCKET_NAME")
file_name = f"sf_police_report/{datetime.date.today()}.json"

if None in (service_account_key, project_id, bucket_name):
    raise ValueError("One or more required GCP environment variables are not set.")

store_to_gcs(
    service_account_key,
    project_id,
    bucket_name,
    file_name,
    json.dumps(data, indent=4),
)

✅ Successfully uploaded 'sf_police_report/2025-09-08.json' to bucket 'msds692-dataacq-ttj'.


Complete retrieve_data_from_gcs() to return a list of list which includes values corresponding to keys in key_list.
If

```
key_list = ['incident_datetime', 'report_datetime', 'incident_code',
            'incident_category', 'incident_description', 'latitude',
            'longitude', 'police_district']
```

This should return

```
[['2025-08-26T23:17:00.000',
  '2025-08-26T23:17:00.000',
  '07041',
  'Recovered Vehicle',
  'Vehicle, Recovered, Auto',
  None,
  None,
  'Out of SF'],
 ['2025-08-27T00:37:00.000',
  '2025-08-27T00:37:00.000',
  '04134',
  'Assault',
  'Battery',
  '37.78041458129883',
  '-122.44901275634766',
  'Park'],....]
```

If a key doesn't exist, its value should be `None`


In [9]:
def retrieve_data_from_gcs(
    service_account_key: str,
    project_id: str,
    bucket_name: str,
    file_name: str,
    key_list: list,
) -> list:
    credentials = service_account.Credentials.from_service_account_file(
        service_account_key,
    )
    client = storage.Client(project=project_id, credentials=credentials)
    bucket = client.bucket(bucket_name)
    file = bucket.blob(file_name)
    content = json.loads(file.download_as_string())

    output = []
    for data in content:
        row = []
        for key in key_list:
            row.append(data.get(key, None))
        output.append(row)

    return output


In [10]:
key_list = [
    "incident_datetime",
    "report_datetime",
    "incident_code",
    "incident_category",
    "incident_description",
    "latitude",
    "longitude",
    "police_district",
]
data = retrieve_data_from_gcs(
    service_account_key,
    project_id,
    bucket_name,
    f"sf_police_report/{datetime.date.today()}.json",
    key_list,
)

In [11]:
data[:6]

[['2025-06-13T12:41:00.000',
  '2025-06-13T12:46:00.000',
  '04134',
  'Assault',
  'Battery',
  '37.7181282043457',
  '-122.41417694091797',
  'Ingleside'],
 ['2025-05-21T00:00:00.000',
  '2025-05-21T08:15:00.000',
  '06224',
  'Larceny Theft',
  'Theft, From Unlocked Vehicle, >$950',
  None,
  None,
  'Ingleside'],
 ['2025-06-11T08:00:00.000',
  '2025-06-12T11:27:00.000',
  '05043',
  'Burglary',
  'Burglary, Residence, Unlawful Entry',
  '37.77315139770508',
  '-122.42222595214844',
  'Northern'],
 ['2025-06-12T00:09:00.000',
  '2025-06-12T00:09:00.000',
  '03014',
  'Robbery',
  'Robbery, Street or Public Place, W/ Force',
  '37.761837005615234',
  '-122.41935729980469',
  'Mission'],
 ['2025-05-23T00:00:00.000',
  '2025-06-12T12:24:00.000',
  '09027',
  'Fraud',
  'False Personation',
  '37.734825134277344',
  '-122.38738250732422',
  'Bayview'],
 ['2025-06-13T02:55:00.000',
  '2025-06-13T02:55:00.000',
  '07041',
  'Recovered Vehicle',
  'Vehicle, Recovered, Auto',
  None,
  None